# RetailHero Data Understanding and Exploratory Data Analysis

This notebook examines the raw RetailHero tables before feature engineering
or model training.

The notebook focuses on:

- raw dataset inventory
- table schemas and data types
- table grain and business meaning
- primary-key and relationship checks
- missing values and duplicate records
- client characteristics
- treatment and outcome distributions
- purchase-history structure
- product-data structure
- data-quality issues that affect feature engineering

The competition test set is not used for exploratory analysis, feature
decisions, model training, or validation.

In [32]:
from pathlib import Path
import sys

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

In [33]:
PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "configs" / "data.yaml").exists()
)

SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

PROJECT_ROOT

WindowsPath('d:/thao/d/uplif_model/uplif_customer_selection')

In [34]:
from uplift_modeling.data.retailhero import (
    get_retailhero_paths,
)

In [35]:
config_path = PROJECT_ROOT / "configs" / "data.yaml"

with config_path.open("r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

retailhero_config = config["retailhero"]
retailhero_config

{'raw_path': 'data/raw/retailhero',
 'processed_path': 'data/processed/retailhero',
 'files': {'clients': 'clients.csv',
  'products': 'products.csv',
  'purchases': 'purchases.csv',
  'uplift_train': 'uplift_train.csv',
  'uplift_test': 'uplift_test.csv',
  'sample_submission': 'uplift_sample_submission.csv'}}

In [36]:
RAW_DATA_DIR = PROJECT_ROOT / retailhero_config["raw_path"]
PROCESSED_DATA_DIR = PROJECT_ROOT / retailhero_config["processed_path"]

PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

data_paths = get_retailhero_paths(RAW_DATA_DIR)

data_paths

{'clients': WindowsPath('d:/thao/d/uplif_model/uplif_customer_selection/data/raw/retailhero/clients.csv'),
 'products': WindowsPath('d:/thao/d/uplif_model/uplif_customer_selection/data/raw/retailhero/products.csv'),
 'purchases': WindowsPath('d:/thao/d/uplif_model/uplif_customer_selection/data/raw/retailhero/purchases.csv'),
 'uplift_train': WindowsPath('d:/thao/d/uplif_model/uplif_customer_selection/data/raw/retailhero/uplift_train.csv'),
 'uplift_test': WindowsPath('d:/thao/d/uplif_model/uplif_customer_selection/data/raw/retailhero/uplift_test.csv'),
 'sample_submission': WindowsPath('d:/thao/d/uplif_model/uplif_customer_selection/data/raw/retailhero/uplift_sample_submission.csv')}

In [37]:
duckdb_connection = duckdb.connect(
    database=":memory:"
)


def to_sql_path(path: Path) -> str:
    """Return a normalized path safe for SQL literals."""
    return (
        path.resolve()
        .as_posix()
        .replace("'", "''")
    )


for table_name, file_path in data_paths.items():
    duckdb_connection.execute(
        f"""
        CREATE OR REPLACE VIEW {table_name} AS
        SELECT *
        FROM read_csv(
            '{to_sql_path(file_path)}',
            header = true
        )
        """
    )

In [38]:
table_shapes = duckdb_connection.execute(
    """
    SELECT 'clients' AS table_name, COUNT(*) AS row_count FROM clients
    UNION ALL
    SELECT 'products', COUNT(*) FROM products
    UNION ALL
    SELECT 'purchases', COUNT(*) FROM purchases
    UNION ALL
    SELECT 'sample_submission', COUNT(*) FROM sample_submission
    UNION ALL
    SELECT 'uplift_train', COUNT(*) FROM uplift_train
    UNION ALL
    SELECT 'uplift_test', COUNT(*) FROM uplift_test
    """
).df()

table_shapes

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,table_name,row_count
0,clients,400162
1,products,43038
2,purchases,45786568
3,sample_submission,200123
4,uplift_train,200039
5,uplift_test,200123


## RetailHero raw database overview

RetailHero is a multi-table retail customer dataset. The raw data is organized around customers, products, transactions, and uplift campaign labels.

### Tables

| Table | Grain | Main role |
|---|---|---|
| `clients` | One row per client | Customer profile and loyalty-card dates |
| `products` | One row per product | Product metadata used to describe purchased items |
| `purchases` | One row per purchased product line within a transaction | Historical purchase behavior before feature engineering |
| `uplift_train` | One row per labeled client | Treatment assignment and observed target for uplift modeling |
| `uplift_test` | One row per unlabeled client | Competition/inference clients without observed target |
| `uplift_sample_submission` | One row per test client | Submission template for the competition test set |

### Important column meanings

#### `clients`

| Column | Meaning |
|---|---|
| `client_id` | Unique customer identifier used to join with purchases and uplift tables |
| `first_issue_date` | Date when the customer first received or activated the loyalty card |
| `first_redeem_date` | Date when the customer first redeemed points or rewards |
| Other client columns | Customer-level attributes available before feature engineering |

#### `products`

| Column | Meaning |
|---|---|
| `product_id` | Unique product identifier used to join with purchases |
| Product hierarchy columns | Product category / segment / hierarchy attributes |
| Product attribute columns | Metadata describing product characteristics |

#### `purchases`

| Column | Meaning |
|---|---|
| `client_id` | Customer who made the purchase |
| `transaction_id` | Transaction or basket identifier |
| `transaction_datetime` | Timestamp of the purchase transaction |
| `regular_points_received` | Regular loyalty points earned from the transaction line |
| `express_points_received` | Express or bonus points earned from the transaction line |
| `regular_points_spent` | Regular loyalty points spent |
| `express_points_spent` | Express or bonus points spent |
| `purchase_sum` | Monetary purchase amount |
| `store_id` | Store where the transaction happened |
| `product_id` | Product purchased |
| `product_quantity` | Quantity of product purchased |
| `trn_sum_from_iss` | Transaction amount measured from issue-side source |
| `trn_sum_from_red` | Transaction amount measured from redemption-side source |

#### `uplift_train`

| Column | Meaning |
|---|---|
| `client_id` | Customer identifier |
| `treatment_flg` | Whether the customer received treatment / campaign exposure |
| `target` | Observed outcome after treatment/control assignment |

#### `uplift_test`

| Column | Meaning |
|---|---|
| `client_id` | Customer identifier for inference or competition submission |

#### `uplift_sample_submission`

| Column | Meaning |
|---|---|
| `client_id` | Test customer identifier |
| Prediction column | Placeholder column for the model output required by the competition format |

